# Medical Screening Expert System for Chronic Kidney Disease
CIS313 Semester Assignment, Topic #11. Dataset: Chronic Kidney Disease (UCI Machine Learning Repository).

In [ ]:
# Information

name = "A. Islam (Aadil)"
student_id = "242-16-025"
section = "20_A"
project_title = "Medical Screening Expert System for Chronic Kidney Disease"

print("===================================")
print("        PROJECT INFORMATION")
print("===================================")
print(f"Name          : {name}")
print(f"ID            : {student_id}")
print(f"Section       : {section}")
print(f"Project Title : {project_title}")
print("===================================")

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")

## 2. Load Dataset (Q3)

In [ ]:
file_path = "data/chronic_kidney_disease.csv"

# If the file is not on this computer (for example on Kaggle or Colab),
# read the same file directly from the UCI website
if not os.path.exists(file_path):
    file_path = "https://archive.ics.uci.edu/static/public/336/data.csv"

df = pd.read_csv(file_path)

df.head()

## 3. Explore Dataset (Q3)

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
# Feature summary table
df.describe().round(2)

`random_state` fixes the "random" choice so the same rows are picked every time the notebook runs.

In [ ]:
df.sample(5, random_state=42)

*(Write your interpretation here.)*

## 4. Data Preprocessing (Q3)
### 4.1 Clean the text values

Some text values have a hidden tab character (for example `"ckd\t"` and `"\tno"`), so the same answer looks like two different values. `str.strip()` removes the extra spaces and tabs.

In [ ]:
# Columns that hold text (the other columns hold numbers)
text_columns = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane", "class"]
numeric_columns = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot",
                   "hemo", "pcv", "wbcc", "rbcc"]

print("Before cleaning:", df["class"].unique())

for col in text_columns:
    df[col] = df[col].str.strip()

print("After cleaning :", df["class"].unique())

*(Write your interpretation here.)*

### 4.2 Missing values and duplicates

In [ ]:
# Check for missing values and duplicate rows
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Rows with no missing value at all:", len(df.dropna()))

In [ ]:
# Chart: number of missing values in each column
missing_counts = df.isnull().sum().sort_values(ascending=False)

plt.figure(figsize=(12, 4))
sns.barplot(x=missing_counts.index, y=missing_counts.values, color="steelblue")
plt.title("Missing Values per Column")
plt.xlabel("Column")
plt.ylabel("Number of missing values")
plt.xticks(rotation=45)
plt.show()

Removing every row with a missing value would throw away most of the patients, so the missing values are filled instead: the median for number columns and the most common value (mode) for text columns.

In [ ]:
# Number columns: fill with the median of the column
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

# Text columns: fill with the most common value of the column
for col in text_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values left:", df.isnull().sum().sum())

*(Write your interpretation here.)*

### 4.3 Create the target
The `class` column is text. It is changed to numbers: 1 = chronic kidney disease (ckd), 0 = no chronic kidney disease (notckd).

In [ ]:
df["target"] = (df["class"] == "ckd").astype(int)

# The original column is no longer needed (keeping it would give the answer away)
df = df.drop(columns="class")

df["target"].value_counts()

*(Write your interpretation here.)*

## 5. Data Visualization (Q3)
### Chart 1: Class distribution

In [ ]:
# Number of patients in each class
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target")
plt.title("Class Distribution")
plt.xlabel("Target (0 = no kidney disease, 1 = kidney disease)")
plt.ylabel("Number of patients")
plt.show()

*(Write your interpretation here.)*

### Chart 2: Correlation heatmap

In [ ]:
# Correlation between the number columns and the target
plt.figure(figsize=(13, 10))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Numeric Features and Target")
plt.show()

In [ ]:
# Correlation of each numeric feature with the target, from highest to lowest
df.corr(numeric_only=True)["target"].drop("target").sort_values(ascending=False).round(2)

*(Write your interpretation here.)*

### Chart 3: Hemoglobin and serum creatinine by target

In [ ]:
# How two blood test results differ between the two classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["hemo", "sc"]):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
    ax.set_title(f"{col} by target")
    ax.set_xlabel("Target (0 = no kidney disease, 1 = kidney disease)")

plt.tight_layout()
plt.show()

In [ ]:
# Average hemoglobin and serum creatinine in each class
df.groupby("target")[["hemo", "sc"]].mean().round(2)

*(Write your interpretation here.)*

## 6. Encode, Split and Scale (Q3)

The text columns are changed to 0/1 columns with one-hot encoding. Every text column has only two values (for example yes/no), so one 0/1 column is enough for each (`drop_first=True`).

In [ ]:
# All text columns except the target
categorical_columns = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane"]

df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True, dtype=int)

df_encoded.head()

In [ ]:
X = df_encoded.drop(columns="target")
y = df_encoded["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

80% of the data is used for training and 20% for testing.
`stratify=y` keeps the same proportion of each class in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

In [ ]:
# Keep the original (unscaled) age of the test patients for Section 11
age_test = X_test["age"]

In [ ]:
# Scale the features (fit only on training data to avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

*(Write your interpretation here.)*

## 7. Train Models (Q4)

Five different models are trained on the same training data:

| Model | How it works | Main hyperparameter |
|---|---|---|
| Logistic Regression | Weighted sum of the features, turned into a probability | `C` |
| KNN | Looks at the K most similar patients and takes a vote | `n_neighbors` |
| Decision Tree | Asks a chain of yes/no questions, like the rules of an expert system | `max_depth` |
| Naive Bayes | Calculates the probability of each class and picks the larger one | `var_smoothing` |
| Random Forest | Many decision trees voting together | `n_estimators` |

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"{name} trained")

### Rules of the expert system
A decision tree is a set of IF-THEN rules. The same tree is trained here on the unscaled values, so the rules show real medical units (class 1 = kidney disease, class 0 = no kidney disease).

In [ ]:
rule_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
rule_tree.fit(X_train, y_train)

print(export_text(rule_tree, feature_names=list(X.columns)))

*(Write your interpretation here.)*

## 8. Model Outputs (Q5)

A large gap between training accuracy and test accuracy is a sign of overfitting.

In [ ]:
# Predictions of every model on the test set
predictions = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test_scaled)

# Training accuracy vs test accuracy
accuracy_rows = []

for name, model in models.items():
    train_pred = model.predict(X_train_scaled)

    accuracy_rows.append({
        "Model": name,
        "Training Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, predictions[name]),
    })

pd.DataFrame(accuracy_rows).round(3)

In [ ]:
# Confusion matrix of every model
class_names = ["No CKD", "CKD"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, name in zip(axes, models):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

# There are 5 models, so the 6th box is left empty
axes[5].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Classification report of every model
for name in models:
    print("=====", name, "=====")
    print(classification_report(y_test, predictions[name], target_names=class_names))

*(Write your interpretation here.)*

## 9. Evaluate and Compare Models (Q6)

ROC-AUC needs the predicted probability of class 1 (kidney disease), not the 0/1 prediction, so `predict_proba(...)[:, 1]` is used for it.

In [ ]:
results = []

for name, model in models.items():
    y_pred = predictions[name]
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).sort_values(by=["Accuracy", "F1 Score"], ascending=False)
results_df.round(3).reset_index(drop=True)

In [ ]:
# Model comparison chart
results_df.plot.bar(x="Model", y=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
                    figsize=(12, 5))
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("")
plt.xticks(rotation=0)
plt.title("Test Scores by Model")
plt.legend(loc="lower right")
plt.show()

*(Write your interpretation here.)*

## 10. Hyperparameter Tuning (Q7)

Two hyperparameters of KNN are tuned:
- `n_neighbors` (K): how many nearest patients vote.
- `weights`: `uniform` gives every neighbour an equal vote, `distance` gives closer neighbours a stronger vote.

Each pair is scored with 5-fold cross-validation on the training data, so the test set is not used for tuning.

In [ ]:
# K is the number of nearest patients that vote (n_neighbors in scikit-learn)
k_values = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 25]
weight_values = ["uniform", "distance"]

tuning_results = []

for weights in weight_values:
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k, weights=weights)
        cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)

        tuning_results.append({
            "n_neighbors": k,
            "weights": weights,
            "CV Accuracy": cv_scores.mean(),
        })

tuning_df = pd.DataFrame(tuning_results)

# Table: one row for each K, one column for each weights option
tuning_table = tuning_df.pivot(index="n_neighbors", columns="weights", values="CV Accuracy")
tuning_table.round(3)

In [ ]:
# Chart: K on the x-axis, one line for each weights option
plt.figure(figsize=(10, 5))

for weights in weight_values:
    part = tuning_df[tuning_df["weights"] == weights]
    plt.plot(part["n_neighbors"], part["CV Accuracy"], marker="o", label=f"weights = {weights}")

plt.xticks(k_values)
plt.xlabel("K (n_neighbors)")
plt.ylabel("CV Accuracy")
plt.title("KNN Tuning: CV Accuracy for Each K and weights")
plt.legend()
plt.show()

In [ ]:
# Best pair = the row with the highest CV accuracy
best_row = tuning_df.loc[tuning_df["CV Accuracy"].idxmax()]
best_k = int(best_row["n_neighbors"])
best_weights = best_row["weights"]

print("Best K (n_neighbors):", best_k)
print("Best weights        :", best_weights)
print("Best CV accuracy    :", round(best_row["CV Accuracy"], 3))

In [ ]:
# Train KNN again with the best pair and test it
best_knn = KNeighborsClassifier(n_neighbors=best_k, weights=best_weights)
best_knn.fit(X_train_scaled, y_train)

best_pred = best_knn.predict(X_test_scaled)
best_prob = best_knn.predict_proba(X_test_scaled)[:, 1]

# The KNN from Section 7 (n_neighbors=5, weights="uniform") for comparison
default_pred = predictions["KNN"]
default_prob = models["KNN"].predict_proba(X_test_scaled)[:, 1]

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "KNN before tuning": [
        accuracy_score(y_test, default_pred),
        precision_score(y_test, default_pred),
        recall_score(y_test, default_pred),
        f1_score(y_test, default_pred),
        roc_auc_score(y_test, default_prob),
    ],
    "KNN after tuning": [
        accuracy_score(y_test, best_pred),
        precision_score(y_test, best_pred),
        recall_score(y_test, best_pred),
        f1_score(y_test, best_pred),
        roc_auc_score(y_test, best_prob),
    ],
})

comparison.round(3)

*(Write your interpretation here.)*

## 11. Performance Across Groups (Q10)

The tuned KNN is checked separately for younger and older patients. The number of patients in each group is shown too, because a score from a small group is less reliable.

In [ ]:
# One table with the age, the real answer and the prediction of the tuned KNN
test_results = pd.DataFrame({
    "age": age_test,
    "actual": y_test,
    "predicted": best_pred,
})

groups = {
    "Age below 50": test_results[test_results["age"] < 50],
    "Age 50 or above": test_results[test_results["age"] >= 50],
}

group_results = []

for group_name, group in groups.items():
    group_results.append({
        "Group": group_name,
        "Patients": len(group),
        "With kidney disease": group["actual"].sum(),
        "Accuracy": accuracy_score(group["actual"], group["predicted"]),
        "Recall": recall_score(group["actual"], group["predicted"], zero_division=0),
    })

pd.DataFrame(group_results).round(3)

*(Write your interpretation here.)*

## 12. Explainability: Feature Importance (Q10)

KNN has no built-in feature importance, so permutation importance is used: the values of one feature are shuffled, and the drop in test accuracy shows how much the model depends on that feature.

In [ ]:
# Permutation importance of the tuned KNN on the test set
result = permutation_importance(best_knn, X_test_scaled, y_test,
                                n_repeats=30, random_state=42)

feature_importance = pd.Series(
    result.importances_mean, index=X.columns
).sort_values(ascending=False)

feature_importance.round(3)

In [ ]:
# Feature importance chart
plt.figure(figsize=(8, 7))
sns.barplot(x=feature_importance.values, y=feature_importance.index, color="seagreen")
plt.xlabel("Drop in accuracy when the feature is shuffled")
plt.title("Permutation Feature Importance (Tuned KNN)")
plt.show()

*(Write your interpretation here.)*